In [ ]:
import cv2
import numpy as np
from tensorflow import keras
from collections import deque


model = keras.models.load_model('asl_cnn_model.pth')

labels = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ") + ["del", "nothing", "space"]

IMG_SIZE = (32, 32)
CONFIDENCE_THRESHOLD = 0.7  

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

print("Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Error: Could not read frame.")
        break

    frame = cv2.flip(frame, 1)

    roi_start, roi_end = (100, 100), (400, 400)
    roi = frame[roi_start[1]:roi_end[1], roi_start[0]:roi_end[0]]
    cv2.rectangle(frame, roi_start, roi_end, (255, 0, 0), 2)

    roi_resized = cv2.resize(roi, IMG_SIZE)
    roi_normalized = roi_resized / 255.0
    roi_reshaped = np.expand_dims(roi_normalized, axis=0)

    predictions = model.predict(roi_reshaped, verbose=0)
    predicted_index = np.argmax(predictions)
    confidence = predictions[0][predicted_index]
    predicted_label = labels[predicted_index]
    prediction_queue = []

    if confidence > CONFIDENCE_THRESHOLD:
        prediction_queue.append(predicted_label)
        smoothed_prediction = max(set(prediction_queue), key=prediction_queue.count)
    else:
        smoothed_prediction = "Unknown"

    cv2.putText(frame, f"Prediction: {smoothed_prediction}", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Confidence: {confidence:.2f}", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    cv2.imshow("ASL Gesture Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()


2025-06-19 08:43:19.758258: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/opt/miniconda3/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:719: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 12 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Press 'q' to quit.


: 